In [21]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import importlib
from pathlib import Path
from dataclasses import dataclass, field
from typing import Any, Optional, List, Dict, Tuple, Union, Callable, Iterable, Set

import os
import copy
import time
import gc
from collections import Counter
from queue import PriorityQueue

from pytket._tket.circuit import Circuit, OpType
from pytket.circuit.display import render_circuit_jupyter
from pytket.qasm import circuit_from_qasm
import json
import pytket
import numpy as np
from pytket.passes import EulerAngleReduction
from pytket import OpType   

from tket2.rewrite import default_ecc_rewriter, ECCRewriter, CircuitRewrite
from tket2.circuit import Tk2Circuit, T2Op, to_hugr_dot

In [2]:
invalid_file = "../test_files/invalid.qasm"

invalid_circ = circuit_from_qasm(invalid_file)
#with open(invalid_file, "r") as f:
#    circ = Circuit.from_dict(json.load(f))

eccs = default_ecc_rewriter()

In [24]:
class Work:
    cost: "Cost"
    circ: Circuit
    prev_circ: Optional[Circuit]

    def __init__(self, circ: Circuit, prev_circ: Circuit = None):
        self.cost = Cost.from_tk1_circ(circ)
        self.circ = circ
        self.prev_circ = prev_circ

    def __lt__(self, other):
        return self.cost < other.cost

@dataclass
class Cost:
    cz_count: int
    gate_count: int

    def __add__(self, other):
        return Cost(self.cz_count + other.cz_count, self.gate_count + other.gate_count)

    def __lt__(self, other):
        return (self.cz_count, self.gate_count) < (other.cz_count, other.gate_count)

    def __le__(self, other):
        return (self.cz_count, self.gate_count) <= (other.cz_count, other.gate_count)

    @staticmethod
    def from_tk1_circ(circ: Circuit) -> "Cost":
        num_cz = circ.n_gates_of_type(OpType.CZ)
        num_gates = circ.n_gates
        return Cost(num_cz, num_gates)

    @staticmethod
    def from_tk2_circ(circ: Tk2Circuit) -> "Cost":
        return circ.circuit_cost(Cost.from_op)

    @staticmethod
    def from_op(op: T2Op) -> "Cost":
        return Cost(int(op.type == OpType.CZ), 1)

def manual_badger(circ, eccs) -> bool:
    max_queue_size = 100
    queue = PriorityQueue(maxsize=max_queue_size)
    seen_hashes = set()
    initial_unitary = circ.get_unitary()
    initial_work = Work(circ)
    global_best = initial_work.cost

    print("Initial cost:", global_best)

    queue.put(initial_work)

    last_circ_time = None

    while not queue.empty():
        last_circ_time = time.time()
        gc.collect()

        work = queue.get_nowait()
        circ_cost = work.cost
        circ_tk1 = work.circ
        circ = Tk2Circuit(circ_tk1)

        print(f"Processing new circ. Queue size {queue.qsize()}. Seen hashes {len(seen_hashes)}. Circ CZ count {circ_tk1.n_gates_of_type(OpType.CZ)}. Circ gate count {circ_tk1.n_gates}.")


        unitary = circ_tk1.get_unitary()
        if not np.allclose(unitary, initial_unitary):
            print("Unitary changed. See 'pre-circ.json' and 'post-circ.json'")
            json.dump(work.prev_circ.to_dict(), open("pre-circ.json", "w"))
            json.dump(circ_tk1.to_tket1_json(), open("post-circ.json", "w"))
            #json.dump(rewrite.replacement().to_tket1_json(), open("rewrite.json", "w"))
            with open("pre-circ.dot", "w") as f:
                f.write(to_hugr_dot(work.prev_circ))
            with open("post-circ.dot", "w") as f:
                f.write(to_hugr_dot(circ_tk1))
            #with open("rewrite.dot", "w") as f:
            #    f.write(to_hugr_dot(rewrite.replacement()))
            return False
        del unitary

        for rewrite in eccs.get_rewrites(circ):
            new_circ = copy.copy(circ)
            new_circ.apply_rewrite(rewrite)

            new_circ_tk1 = new_circ.to_tket1()
            new_cost = Cost.from_tk1_circ(new_circ_tk1)

            if circ_cost < new_cost:
                del new_circ
                del new_circ_tk1
                continue

            new_circ_hash = new_circ.hash()
            if new_circ_hash in seen_hashes:
                del new_circ
                del new_circ_tk1
                continue
            seen_hashes.add(new_circ_hash)

            if new_cost < global_best:
                print("New best:", global_best)
                global_best = new_cost

            if queue.full():
                elems = [queue.get_nowait() for _ in range(queue.qsize() // 2)]
                while not queue.empty():
                    queue.get_nowait()
                for e in elems:
                    queue.put_nowait(e)
                del elems

            queue.put_nowait(Work(new_circ_tk1, circ_tk1))
            del new_circ
            del new_circ_tk1

        print(f"Processing the circ took {time.time() - last_circ_time} seconds")

    print("The optimisation finished correctly")
    return True


In [25]:
manual_badger(invalid_circ, eccs)

Initial cost: Cost(cz_count=48, gate_count=400)
Processing new circ. Queue size 0. Seen hashes 0. Circ CZ count 48. Circ gate count 400.
New best: Cost(cz_count=48, gate_count=400)
Processing the circ took 3.7730934619903564 seconds
Processing new circ. Queue size 55. Seen hashes 106. Circ CZ count 48. Circ gate count 399.
New best: Cost(cz_count=48, gate_count=399)
Processing the circ took 4.166179656982422 seconds
Processing new circ. Queue size 58. Seen hashes 210. Circ CZ count 48. Circ gate count 398.
New best: Cost(cz_count=48, gate_count=398)
Processing the circ took 4.4429662227630615 seconds
Processing new circ. Queue size 59. Seen hashes 312. Circ CZ count 48. Circ gate count 397.
New best: Cost(cz_count=48, gate_count=397)
Processing the circ took 4.423537015914917 seconds
Processing new circ. Queue size 59. Seen hashes 413. Circ CZ count 48. Circ gate count 396.
New best: Cost(cz_count=48, gate_count=396)
